In [ ]:
import os
import json
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

/Users/robinchriqui/project/LLM_projects/conv_classifiation
['dataset_vip.jsonl', 'requirements.txt', 'llm_judge.ipynb', 'Take-Home Assignment — Senior AI Engineer.pdf', 'casino_signal_dataset_realistic_300.jsonl', 'readme.md', 'dataset.jsonl', 'conv.ipynb']


In [ ]:
with open("dataset_vip.jsonl", "r", encoding="utf-8") as f:
    first_line = f.readline()

print(repr(first_line))

'{"id":"ex_001","conversation":["Hi we\'re thinking about coming back in March."],"signals":[{"category":"intent","value":"trip_planning"}]}\n'


# 1) Calls the models

Budget: limited preference for low cost models intelligence
MimoV2-flash
Minimax-M2.5
Deepseek V3.2
Grok 4.1 fast


deepseek/deepseek-v3.2
x-ai/grok-4.1-fast
xiaomi/mimo-v2-flash

openai/gpt-oss-120b 
$0,039/M input tokens
$0,19/M output tokens


## openai/gpt-5-nano
$0,05/M input tokens
$0,40/M output tokens

##google/gemini-3.1-flash-lite-preview
$0,25/M input tokens
$1,50/M output tokens
$0,50/M audio tokens


In [ ]:
competitors = []
answers = []

In [ ]:
import os
import json
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

# Will hold results per model after running main()
model_results = {}
last_run_results = None

competitors = [
    #"qwen/qwen3.5-flash-02-23",
    "google/gemini-3.1-flash-lite-preview",
    "xiaomi/mimo-v2-flash",
    "deepseek/deepseek-v3.2",
    "x-ai/grok-4.1-fast",
]

ALLOWED_LABELS = {
    "intent": {"trip_planning", "room_booking", "dining_booking"},
    "value": {"suite_preference", "high_budget", "large_group", "vip_expectation"},
    "sentiment": {"positive_experience", "negative_experience"},
    "life_event": {"birthday", "anniversary", "honeymoon", "promotion", "celebration"},
    "competitive": {
        "competitor_wynn",
        "competitor_cosmo",
        "competitor_bellagio",
        "competitor_offer",
    },
}

SYSTEM_PROMPT = """
You are an information extraction system for VIP casino guest conversations.

Extract all signals present in the conversation.

Allowed categories and values:

intent: trip_planning, room_booking, dining_booking
value: suite_preference, high_budget, large_group, vip_expectation
sentiment: positive_experience, negative_experience
life_event: birthday, anniversary, honeymoon, promotion, celebration
competitive: competitor_wynn, competitor_cosmo, competitor_bellagio, competitor_offer

Return JSON only in this format:
{
  "signals": [
    {"category": "intent", "value": "trip_planning"}
  ]
}

If no signal is present, return:
{"signals": []}
""".strip()


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def safe_parse_json(text):
    text = text.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")
        if start != -1 and end != -1:
            try:
                return json.loads(text[start:end+1])
            except json.JSONDecodeError:
                pass
    return {"signals": []}


def normalize_signals(signals):
    out = set()
    if not isinstance(signals, list):
        return out
    for s in signals:
        if not isinstance(s, dict):
            continue
        c = s.get("category")
        v = s.get("value")
        if c in ALLOWED_LABELS and v in ALLOWED_LABELS[c]:
            out.add((c, v))
    return out


def build_prompt(conversation):
    return "Conversation:\n" + "\n".join(f"- {x}" for x in conversation) + "\n\nReturn the JSON now."


def call_model(model_name, conversation):
    response = client.chat.completions.create(
        model=model_name,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_prompt(conversation)},
        ],
    )
    # Some providers may return None for content on error; guard against that
    text = response.choices[0].message.content or ""
    parsed = safe_parse_json(text) or {"signals": []}
    return {
        "raw_text": text,
        "signals": [
            {"category": c, "value": v}
            for c, v in sorted(normalize_signals(parsed.get("signals", [])))
        ],
    }


def compute_prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"precision": round(p, 4), "recall": round(r, 4), "f1": round(f1, 4)}


def score_predictions(gold_rows, pred_rows):
    gold_by_id = {x["id"]: x for x in gold_rows}
    pred_by_id = {x["id"]: x for x in pred_rows}

    tp = fp = fn = 0

    for row_id, gold in gold_by_id.items():
        pred = pred_by_id.get(row_id, {"signals": []})

        gold_set = normalize_signals(gold["signals"])
        pred_set = normalize_signals(pred["signals"])

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    return compute_prf(tp, fp, fn)


def main():
    global last_run_results, model_results
    #dataset = load_jsonl("dataset.jsonl")
    dataset=load_jsonl('dataset_vip.jsonl')

    leaderboard = []

    for model_name in competitors:
        # Avoid re-running models we've already evaluated in this kernel session
        if model_name in model_results:
            print(f"\nSkipping {model_name} (already in model_results)")
            scores = model_results[model_name]["metrics"]
            leaderboard.append({
                "model": model_name,
                "f1": scores["f1"],
                "precision": scores["precision"],
                "recall": scores["recall"],
            })
            continue

        print(f"\nTesting {model_name}")
        predictions = []

        for row in dataset:
            pred = call_model(model_name, row["conversation"])
            predictions.append({
                "id": row["id"],
                "signals": pred["signals"],
                "raw_text": pred["raw_text"],
            })

        scores = score_predictions(dataset, predictions)
        print("Scores:", scores)

        # Store per-model metrics and predictions so we can visualize later
        model_results[model_name] = {
            "metrics": scores,
            "predictions": predictions,
        }

        leaderboard.append({
            "model": model_name,
            "f1": scores["f1"],
            "precision": scores["precision"],
            "recall": scores["recall"],
        })

    leaderboard = sorted(leaderboard, key=lambda x: x["f1"], reverse=True)

    results = []
    for row in leaderboard:
        idx = competitors.index(row["model"]) + 1
        results.append(str(idx))

    results_dict = {
        "results": results,
        "metrics": {
            row["model"]: {
                "precision": row["precision"],
                "recall": row["recall"],
                "f1": row["f1"],
            }
            for row in leaderboard
        }
    }

    # Cache the full JSON result of the last run
    last_run_results = results_dict

    print("\nFinal JSON:")
    print(json.dumps(results_dict, indent=2))

    print("\nRanking:")
    ranks = results_dict["results"]
    for index, result in enumerate(ranks):
        competitor = competitors[int(result)-1]
        print(f"Rank {index+1}: {competitor}")


if __name__ == "__main__":
    main()


Testing google/gemini-3.1-flash-lite-preview
Scores: {'precision': 0.9474, 'recall': 0.9474, 'f1': 0.9474}

Testing xiaomi/mimo-v2-flash
Scores: {'precision': 0.9596, 'recall': 0.8333, 'f1': 0.892}

Testing deepseek/deepseek-v3.2
Scores: {'precision': 0.8261, 'recall': 0.6667, 'f1': 0.7379}

Testing x-ai/grok-4.1-fast
Scores: {'precision': 0.8594, 'recall': 0.9649, 'f1': 0.9091}

Final JSON:
{
  "results": [
    "1",
    "4",
    "2",
    "3"
  ],
  "metrics": {
    "google/gemini-3.1-flash-lite-preview": {
      "precision": 0.9474,
      "recall": 0.9474,
      "f1": 0.9474
    },
    "x-ai/grok-4.1-fast": {
      "precision": 0.8594,
      "recall": 0.9649,
      "f1": 0.9091
    },
    "xiaomi/mimo-v2-flash": {
      "precision": 0.9596,
      "recall": 0.8333,
      "f1": 0.892
    },
    "deepseek/deepseek-v3.2": {
      "precision": 0.8261,
      "recall": 0.6667,
      "f1": 0.7379
    }
  }
}

Ranking:
Rank 1: google/gemini-3.1-flash-lite-preview
Rank 2: x-ai/grok-4.1-fast
Ra

In [ ]:
import gradio as gr
import pandas as pd

# --- Dashboard v3: full per-model view ---

F1_GREEN = 0.9
F1_AMBER = 0.8
PREC_GREEN = 0.9
PREC_AMBER = 0.8
REC_GREEN = 0.9
REC_AMBER = 0.8


def _metric_color_v3(value: float, metric_type: str) -> str:
    if metric_type == "f1":
        if value >= F1_GREEN:
            return "#22c55e"  # green-500
        if value >= F1_AMBER:
            return "#f97316"  # orange-500
        return "#ef4444"      # red-500
    if metric_type == "precision":
        if value >= PREC_GREEN:
            return "#22c55e"
        if value >= PREC_AMBER:
            return "#f97316"
        return "#ef4444"
    if metric_type == "recall":
        if value >= REC_GREEN:
            return "#22c55e"
        if value >= REC_AMBER:
            return "#f97316"
        return "#ef4444"
    return "#e5e7eb"  # gray-200


def _metric_card_v3(label: str, value: float, metric_type: str) -> str:
    color = _metric_color_v3(value, metric_type)
    value_str = f"{value:.4f}"
    return f"""
    <div style="
        padding: 10px 12px;
        border-radius: 10px;
        background-color: #111827;
        border: 1px solid #1f2937;
        box-shadow: 0 1px 3px rgba(0,0,0,0.25);
    ">
        <div style="font-size: 11px; color: #9ca3af; margin-bottom: 4px;">{label}</div>
        <div style="font-size: 20px; font-weight: 600; color: {color};">
            {value_str}
        </div>
    </div>
    """


def _build_data_from_results():
    """Build DataFrames and summary HTML from last_run_results.

    Returns (summary_html, df_bar, df_table, short_to_full_map).

    df_table includes per-category F1 scores so we can see,
    for each model, how well it does on intent / value / etc.
    """
    if last_run_results is None:
        html = (
            "<div style='padding: 20px; text-align: center; color: #9ca3af;'>"
            "Run the evaluation cell (main()) first to populate results."
            "</div>"
        )
        empty = pd.DataFrame(
            {
                "Model": [],
                "Short Model": [],
                "Precision": [],
                "Recall": [],
                "F1": [],
                "Intent": [],
                "Value": [],
                "Sentiment": [],
                "Life Event": [],
                "Competitive": [],
            }
        )
        return html, empty, empty, {}

    metrics = last_run_results.get("metrics", {})
    if not metrics:
        html = (
            "<div style='padding: 20px; text-align: center; color: #9ca3af;'>"
            "No metrics found in last_run_results."
            "</div>"
        )
        empty = pd.DataFrame(
            {
                "Model": [],
                "Short Model": [],
                "Precision": [],
                "Recall": [],
                "F1": [],
                "Intent": [],
                "Value": [],
                "Sentiment": [],
                "Life Event": [],
                "Competitive": [],
            }
        )
        return html, empty, empty, {}

    # Load gold dataset once so we can compute per-category F1 scores
    try:
        gold_rows = load_jsonl("dataset.jsonl")
    except FileNotFoundError:
        gold_rows = []

    gold_by_id = {row["id"]: row for row in gold_rows}

    def _per_category_f1(predictions):
        """Compute F1 per category (intent/value/sentiment/life_event/competitive)."""
        cats = ["intent", "value", "sentiment", "life_event", "competitive"]
        stats = {c: {"tp": 0, "fp": 0, "fn": 0} for c in cats}

        pred_by_id = {p["id"]: p for p in predictions}

        for row_id, gold in gold_by_id.items():
            gold_set = normalize_signals(gold["signals"])
            pred = pred_by_id.get(row_id, {"signals": []})
            pred_set = normalize_signals(pred.get("signals", []))

            for cat in cats:
                gold_vals = {v for (c, v) in gold_set if c == cat}
                pred_vals = {v for (c, v) in pred_set if c == cat}

                tp = len(gold_vals & pred_vals)
                fp = len(pred_vals - gold_vals)
                fn = len(gold_vals - pred_vals)

                stats[cat]["tp"] += tp
                stats[cat]["fp"] += fp
                stats[cat]["fn"] += fn

        def _f1(tp, fp, fn):
            p = tp / (tp + fp) if tp + fp else 0.0
            r = tp / (tp + fn) if tp + fn else 0.0
            return 2 * p * r / (p + r) if p + r else 0.0

        return {c: _f1(v["tp"], v["fp"], v["fn"]) for c, v in stats.items()}

    rows = []
    short_to_full = {}
    for model_name, m in metrics.items():
        short_name = model_name.split("/")[-1]
        short_to_full[short_name] = model_name

        # Default per-category scores
        cat_scores = {"intent": 0.0, "value": 0.0, "sentiment": 0.0, "life_event": 0.0, "competitive": 0.0}

        # If we have stored predictions for this model, compute per-category F1
        if "model_results" in globals() and isinstance(model_results, dict):
            data = model_results.get(model_name)
            if data and "predictions" in data and gold_by_id:
                cat_scores = _per_category_f1(data["predictions"])

        rows.append(
            {
                "Model": model_name,
                "Short Model": short_name,
                "Precision": m.get("precision", 0.0),
                "Recall": m.get("recall", 0.0),
                "F1": m.get("f1", 0.0),
                "Intent": cat_scores.get("intent", 0.0),
                "Value": cat_scores.get("value", 0.0),
                "Sentiment": cat_scores.get("sentiment", 0.0),
                "Life Event": cat_scores.get("life_event", 0.0),
                "Competitive": cat_scores.get("competitive", 0.0),
            }
        )

    df = pd.DataFrame(rows).sort_values("F1", ascending=False).reset_index(drop=True)
    df["Rank"] = df.index + 1

    # Reorder columns for the table view (including per-category scores)
    df_table = df[
        [
            "Rank",
            "Short Model",
            "Model",
            "Precision",
            "Recall",
            "F1",
            "Intent",
            "Value",
            "Sentiment",
            "Life Event",
            "Competitive",
        ]
    ]
    df_bar = df[["Short Model", "F1"]]

    best_row = df.iloc[0]
    best_model = best_row["Model"]
    best_f1 = float(best_row["F1"])
    avg_f1 = float(df["F1"].mean())
    avg_prec = float(df["Precision"].mean())
    avg_rec = float(df["Recall"].mean())

    summary_html = f"""
    <div style="padding: 8px 0 4px 0;">
      <div style="margin-bottom: 16px;">
        <div style="font-size: 13px; color: #9ca3af; margin-bottom: 2px;">Best Model</div>
        <div style="font-size: 22px; font-weight: 700;">{best_model}</div>
      </div>

      <div style="display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 10px;">
        {_metric_card_v3("Best model F1", best_f1, "f1")}
        {_metric_card_v3("Average F1 across models", avg_f1, "f1")}
        {_metric_card_v3("Average Precision", avg_prec, "precision")}
        {_metric_card_v3("Average Recall", avg_rec, "recall")}
      </div>

      <div style="margin-top: 10px; font-size: 10px; color: #6b7280;">
        Colors: green = high, orange = medium, red = lower performance.
      </div>
    </div>
    """

    return summary_html, df_bar, df_table, short_to_full


def build_overview_for_app():
    """Gradio-friendly wrapper: overview HTML, bar DF, table DF, dropdown choices.

    Uses gr.update so it works across Gradio versions.
    """
    summary_html, df_bar, df_table, short_to_full = _build_data_from_results()
    short_names = list(short_to_full.keys())
    dropdown_update = gr.update(
        choices=short_names,
        value=short_names[0] if short_names else None,
    )
    return summary_html, df_bar, df_table, dropdown_update


def model_detail(selected_short: str):
    """Return detailed metrics and sample predictions for a single model."""
    if not selected_short or last_run_results is None:
        html = (
            "<div style='padding: 12px; color: #9ca3af;'>"
            "Select a model and make sure you've run the evaluation cell."
            "</div>"
        )
        return html, pd.DataFrame({"id": [], "signals": []})

    # Rebuild mapping from last_run_results
    metrics = last_run_results.get("metrics", {})
    short_to_full = {name.split("/")[-1]: name for name in metrics.keys()}
    full_name = short_to_full.get(selected_short)
    if full_name is None:
        html = (
            "<div style='padding: 12px; color: #ef4444;'>"
            "No metrics found for the selected model."
            "</div>"
        )
        return html, pd.DataFrame({"id": [], "signals": []})

    m = metrics[full_name]
    prec = float(m.get("precision", 0.0))
    rec = float(m.get("recall", 0.0))
    f1 = float(m.get("f1", 0.0))

    detail_html = f"""
    <div style="padding: 8px 0;">
      <div style="font-size: 14px; color: #9ca3af; margin-bottom: 4px;">Model</div>
      <div style="font-size: 20px; font-weight: 600; margin-bottom: 12px;">{full_name}</div>
      <div style="display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 10px;">
        {_metric_card_v3("Precision", prec, "precision")}
        {_metric_card_v3("Recall", rec, "recall")}
        {_metric_card_v3("F1", f1, "f1")}
      </div>
    </div>
    """

    # Use model_results (if available) to show a light table of predictions
    if "model_results" in globals() and isinstance(model_results, dict):
        data = model_results.get(full_name, {})
        preds = data.get("predictions", [])
        if preds:
            # Keep only a few lightweight columns for display
            rows = []
            for p in preds:
                rows.append({
                    "id": p.get("id"),
                    "signals": p.get("signals"),
                })
            df_preds = pd.DataFrame(rows)
            # Limit number of rows shown so UI stays snappy
            if len(df_preds) > 50:
                df_preds = df_preds.head(50)
            return detail_html, df_preds

    return detail_html, pd.DataFrame({"id": [], "signals": []})


# Build the full app
rag_theme_v3 = gr.themes.Soft(
    primary_hue="indigo",
    font=["Inter", "system-ui", "sans-serif"],
)

with gr.Blocks(
    title="LLM Judge Evaluation Dashboard (all models)",
    theme=rag_theme_v3,
    css=".gradio-container { max-width: 1100px; margin: 0 auto; }",
) as judge_app_v3:
    gr.Markdown("## LLM Judge Evaluation Dashboard")
    gr.Markdown(
        "View metrics for **all models** from the last evaluation run, and inspect per-model details."
    )

    refresh_btn = gr.Button("Refresh from last_run_results", variant="primary")

    with gr.Row():
        with gr.Column(scale=1):
            overview_html = gr.HTML(
                "<div style='padding: 20px; text-align: center; color: #9ca3af;'>"
                "Run the evaluation cell, then click <b>Refresh from last_run_results</b>."
                "</div>"
            )
        with gr.Column(scale=1):
            bar_plot = gr.BarPlot(
                value=None,
                x="Short Model",
                y="F1",
                title="F1 score per model",
                y_lim=[0, 1],
                height=380,
            )

    gr.Markdown("### Model rankings (including per-category F1)")
    models_table = gr.Dataframe(
        headers=[
            "Rank",
            "Short Model",
            "Model",
            "Precision",
            "Recall",
            "F1",
            "Intent",
            "Value",
            "Sentiment",
            "Life Event",
            "Competitive",
        ],
        row_count=(0, "dynamic"),
        col_count=11,
        wrap=True,
    )

    gr.Markdown("### Model details")
    with gr.Row():
        model_dropdown = gr.Dropdown(
            label="Select model",
            choices=[],
            value=None,
        )
        detail_btn = gr.Button("Show details")

    model_detail_html = gr.HTML(
        "<div style='padding: 12px; color: #9ca3af;'>Select a model above to see details.</div>"
    )
    preds_table = gr.Dataframe(
        headers=["id", "signals"],
        row_count=(0, "dynamic"),
        col_count=2,
        wrap=True,
    )

    refresh_btn.click(
        fn=build_overview_for_app,
        inputs=None,
        outputs=[overview_html, bar_plot, models_table, model_dropdown],
    )

    detail_btn.click(
        fn=model_detail,
        inputs=model_dropdown,
        outputs=[model_detail_html, preds_table],
    )

judge_app_v3.launch(inbrowser=True)

/var/folders/5g/jgl3b_zn0l78stgyjbgqvl340000gn/T/ipykernel_6661/4141817702.py:308: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(
/Users/robinchriqui/Desktop/projects/support_alone/venv/lib/python3.11/site-packages/gradio/components/dataframe.py:192: UserWarning: The `col_count` parameter is deprecated and will be removed. Please use `column_count` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.
